# Day 63 · The final run: same test, a stronger model

**First-principles question:** Does your recommendation survive a stronger model, and what does that cost?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

**Order matters:** pick the model, estimate the cost, *then* run. Never the other way round.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import os
import numpy as np
import pandas as pd
from lab import llm
from lab.acme import load_cases, load_results, save_results
from lab.evals import score
from lab.version_b import version_b
from lab.agent import run_agent

assert hasattr(llm, "use"), ("Your lab/llm.py is older than this topic. Copy the new one over it: "
                             "cp ~/FDE-Impromptu/src/assets/learn_starters/p1-06/_track/lab/llm.py lab/llm.py")
cases = load_cases()
by_id = {c["id"]: c for c in cases}
SMALL = llm.CHAT_MODEL
print(len(cases), "cases · small model:", SMALL)

## 1 · Pick the stronger model
- **Free:** a bigger model in your own Ollama, if your Mac has 16 GB of memory or more. Run `ollama pull qwen3:14b`
  in a terminal first.
- **Paid:** `"api:<model name from your provider>"`. Before anything else, set a **hard monthly spending cap** in the
  provider's billing settings. Then put two lines in `.env` in your fde-first-principles folder (it is git-ignored):
  `LLM_BASE_URL=https://…/v1` and `LLM_API_KEY=…`. Never paste the key into a notebook.

In [ ]:
STRONG = "qwen3:14b"          # or "api:<model name>" for a paid model
PAID = STRONG.startswith(llm.API_PREFIX)
print("small:", SMALL, "· strong:", STRONG, "· paid:", PAID)

## 2 · Estimate before you spend
Your day 9 results already know how many tokens one email takes for versions B and C. Multiply by the number of
cases and the price. A paid run must cost less than **a fifth of your cap**: if a bug makes you run it twice, you
are still safe.

In [ ]:
b_small, c_small = load_results("B"), load_results("C")

# Paid model: fill in from the provider's pricing page (US dollars per million tokens). Free model: leave 0.
PRICE_IN_USD, PRICE_OUT_USD, INR_PER_USD = 0.0, 0.0, 85.0
CAP_INR = 500                 # the hard monthly cap you set in the billing settings

def cost_inr(tokens_in, tokens_out):
    return (tokens_in * PRICE_IN_USD + tokens_out * PRICE_OUT_USD) / 1e6 * INR_PER_USD

# TODO 1: expected_inr = the cost of running B once and C once on every case.
#         Use the mean tokens_in and tokens_out per email from b_small and from c_small, times len(cases).
# TODO 2: expected_calls = model calls for that run: one per case for B, plus the mean "steps" of c_small per case for C.
expected_inr = None
expected_calls = None

print(f"Expected cost: ₹{expected_inr:.0f} (cap ₹{CAP_INR}) · about {expected_calls:.0f} model calls")
llm.API_CALL_LIMIT = int(expected_calls * 1.2) + 10     # a runaway loop stops here, long before your cap

## 3 · Paid only: dry-run the API path for free
Your own Ollama speaks the same OpenAI-compatible API as the paid providers. Point the API path at it for 3 cases:
if the answers look like before, the plumbing works and the paid run will not waste money on a bug.
(`qwen3` may "think" first on this path, so it is slower. That is fine for 3 cases.)

In [ ]:
if PAID:
    saved = os.environ.get("LLM_BASE_URL")
    os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"
    llm.use(llm.API_PREFIX + SMALL)
    try:
        for case in cases[:3]:
            out = version_b(case)
            print(case["id"], "→", out.get("intent"), "· expected", case["expected_intent"])
    finally:
        llm.use(SMALL)
        if saved:
            os.environ["LLM_BASE_URL"] = saved
        else:
            os.environ.pop("LLM_BASE_URL", None)
else:
    print("Free model: no dry run needed.")

## 4 · The run: B and C with the stronger model
One line switches the model; everything is cached, so re-running this cell costs nothing. On a local 14B model this
can take an hour: start it, then read the next section while it runs.

In [ ]:
llm.use(STRONG)
try:
    b_strong = [dict(version_b(case), id=case["id"]) for case in cases]
    c_strong = []
    for case in cases:
        answer, trace = run_agent(case)
        c_strong.append(dict(answer, id=case["id"], steps=len(trace),
                             tokens_in=sum(e["tokens_in"] for e in trace),
                             tokens_out=sum(e["tokens_out"] for e in trace),
                             seconds=sum(e["seconds"] for e in trace)))
finally:
    llm.use(SMALL)
save_results("B-strong", b_strong)
save_results("C-strong", c_strong)

actual_inr = sum(cost_inr(r.get("tokens_in", 0), r.get("tokens_out", 0)) for r in b_strong + c_strong)
print(f"Done · this run cost about ₹{actual_inr:.0f} (estimate ₹{expected_inr:.0f}) · API usage: {llm.api_usage()}")

## 5 · Small vs. strong, side by side

In [ ]:
def passed_by_id(rows):
    return {r["id"]: bool(score(by_id[r["id"]], r)["passed"]) for r in rows}

def row(name, rows):
    passed = np.array(list(passed_by_id(rows).values()), dtype=float)
    p, n = passed.mean(), len(passed)
    secs = [r.get("seconds", 0) for r in rows]
    return {"version": name, "pass rate": p, "± 2 se": 2 * np.sqrt(p * (1 - p) / n),
            "p95 s": float(np.percentile(secs, 95, method="inverted_cdf")),
            "₹ per 1,000 emails": 1000 * np.mean([cost_inr(r.get("tokens_in", 0), r.get("tokens_out", 0)) for r in rows])}

table = pd.DataFrame([row("B · small", b_small), row("B · strong", b_strong),
                      row("C · small", c_small), row("C · strong", c_strong)]).set_index("version")
table.round(3)

## 6 · Paired differences: is the stronger model really better?
Both models answered the **same** cases, so compare them case by case. Two averages with overlapping error bars can
hide a real difference; the paired difference does not.

In [ ]:
def paired(small_passed, strong_passed):
    """small_passed and strong_passed: lists of True/False for the same cases, in the same order.
    Returns (mean difference, ± 2 standard errors) of strong − small."""
    # TODO: d = [1 if only strong passed, -1 if only small passed, 0 if they agree] for each case
    #       mean = np.mean(d); ± range = 2 * np.std(d, ddof=1) / np.sqrt(len(d))
    ...

for name, small, strong in [("B", b_small, b_strong), ("C", c_small, c_strong)]:
    s, g = passed_by_id(small), passed_by_id(strong)
    ids = [i for i in s if i in g]
    m, r = paired([s[i] for i in ids], [g[i] for i in ids])
    verdict = "a real difference" if abs(m) > r else "could be noise"
    print(f"{name}: strong − small = {m:+.1%} ± {r:.1%} → {verdict}")

## 7 · Which failures disappear, and which stay?
Compare these case ids with your failure tree (project 6, step 2). Mechanisms like a retrieval miss or missing
context usually survive a bigger model; sampling noise and weak instruction-following often do not.

In [ ]:
for name, small, strong in [("B", b_small, b_strong), ("C", c_small, c_strong)]:
    s, g = passed_by_id(small), passed_by_id(strong)
    fixed = [i for i in s if i in g and g[i] and not s[i]]
    still = [i for i in s if i in g and not g[i] and not s[i]]
    broke = [i for i in s if i in g and s[i] and not g[i]]
    print(f"{name}: fixed by the stronger model {fixed}\n   still failing with both {still}\n   newly broken {broke}")

**Does the recommendation change?** One paragraph, with the numbers: what the stronger model fixes, what it does not fix, and what it costs per 1,000 emails: …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert expected_inr is not None and expected_calls is not None, "Step 2: write the estimate before you run"
assert expected_calls >= len(cases), "Step 2: at least one call per case for B, plus C's steps"
if PAID:
    assert expected_inr <= CAP_INR / 5, "Step 2: a paid run must be estimated under a fifth of your cap"
    assert actual_inr < CAP_INR, "The run cost more than your cap: stop and look at the trace"
assert len(b_strong) == len(cases) and len(c_strong) == len(cases), "Step 4: B and C on every case"
m, r = paired([True, False, False, True], [True, True, False, True])
assert abs(m - 0.25) < 1e-9 and abs(r - 0.5) < 1e-9, "Step 6: paired() should give +25% ± 50% on the example"
assert paired([True] * 4, [True] * 4)[0] == 0, "Step 6: identical results give a difference of 0"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")